# Plant Hydraulic Models

Interactive notebook entry point for the forward and inverse network models. The calculations and plotting functions live in `model.py` and `inverse_model.py`; this notebook imports and calls them rather than duplicating their implementations.

**Boundary-condition note:** the forward model fixes root and leaf pressures. The inverse model fixes root-tip pressure and prescribes leaf-tip flow, solving leaf pressures. These are different boundary-condition setups.

## 1. Imports and configuration

In [ ]:
%matplotlib inline
import importlib
import matplotlib.pyplot as plt
import model
import inverse_model

# Reload after editing the Python modules so the notebook picks up changes.
importlib.reload(model)
importlib.reload(inverse_model)


## 2. Forward model — prescribed terminal pressures

In [ ]:
forward_results = model.solve_network()

print('Forward model: node pressures and hydraulic potentials')
for node in forward_results['nodes']:
    print(f"{node}: z={forward_results['nodes'][node]['z']:.4f} m, "
          f"p={forward_results['pressures'][node]:.6e} Pa, "
          f"phi={forward_results['phi'][node]:.6e} Pa")

print('\\nForward model: internal-node mass conservation')
for item in model.check_mass_conservation(
        forward_results['nodes'], forward_results['branches'], forward_results['flows']):
    if item['is_internal']:
        print(f"{item['node']}: net outflow={item['net_outflow']:.3e} m^3/s, "
              f"relative residual={item['relative_residual']:.3e}, "
              f"{'PASS' if item['passed'] else 'FAIL'}")


In [ ]:
# Forward-model plots: flow structure + two hydraulic-resistance views.
model.plot_flow_results(forward_results)
model.plot_hydraulic_resistance_network(forward_results)
model.plot_hydraulic_resistance_ranking(forward_results)
plt.show()


## 3. Inverse model — prescribed minimum flow at each leaf

In [ ]:
# Q_MIN is the default flow demand per leaf tip, defined in inverse_model.py.
q_min = inverse_model.Q_MIN
inverse_results = inverse_model.solve_for_minimum_leaf_flows(q_min)
inverse_model.print_report(inverse_results, duration=1.0)


In [ ]:
# Inverse-model plots: flow structure + two hydraulic-resistance views.
inverse_model.plot_results(inverse_results)


## 4. Experiment with leaf flow demand

Edit `q_min` below to test another demand per leaf tip. This changes the physical scenario, not just numerical precision. For a linear fixed-resistance network, branch flow scales linearly with the demand and viscous power scales quadratically when all leaf demands are scaled together.

In [ ]:
q_min_test = 2.0 * inverse_model.Q_MIN  # change this multiplier as desired
inverse_test_results = inverse_model.solve_for_minimum_leaf_flows(q_min_test)
inverse_model.print_report(inverse_test_results, duration=1.0)
inverse_model.plot_results(inverse_test_results)
